# The running example's Transformer in NumPy

The Build a Transformer steps follow "the cat sat" through a model with weights chosen by
hand. Here you write that model as general functions: position values for any length and
width, causal attention with any number of heads, the whole forward pass from token IDs to
next-token probabilities, and greedy generation. The checks compare your functions with the
numbers on the step pages and also run them on inputs the pages never show.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It holds the running example's vocabulary and weights, and two helpers
from the steps: a softmax that subtracts each row's largest value first, and LayerNorm with
no learned scale or shift.

In [ ]:
import numpy as np

vocab = ["<end>", "the", "cat", "sat", "on", "mat", "dog", "ran"]

# The running example's weights (Steps 2, 4, 8 and 10).
E = np.array([[0, 0, 0, -1], [1, 0, 0, 0], [0, 1, 1, 0], [0, 0, 1, 1],
              [-1, -1, 0, 1], [1, 1, 0, 0], [0, 1, 0, 1], [1, 0, 1, 0]],
             dtype=float)
W_Q = np.zeros((4, 4))
W_Q[2, 0] = W_Q[3, 1] = W_Q[1, 2] = W_Q[3, 3] = 1
W_K = np.zeros((4, 4))
W_K[1, 0] = W_K[2, 1] = W_K[1, 2] = W_K[3, 3] = 1
W_V = np.eye(4)
W_O = np.eye(4)
W_1 = np.array([[1, 0, 0, -1, 0, 1, 0, 0],
                [0, 1, 0, 0, -1, 0, 1, 0],
                [0, 0, 1, 0, 0, -1, 0, 1],
                [-1, 0, 0, 1, 0, 0, -1, 0]], dtype=float)
W_2 = 0.5 * np.vstack([np.eye(4), -np.eye(4)])


def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    powers = np.exp(shifted)
    return powers / np.sum(powers, axis=axis, keepdims=True)


def layer_norm(z, eps=1e-5):
    mean = z.mean(axis=-1, keepdims=True)
    var = z.var(axis=-1, keepdims=True)
    return (z - mean) / np.sqrt(var + eps)

## Part 1 (coding task): Position values for any length and width

Write `positions(T, d)`, which returns the $T \times d$ table of sinusoidal position values
for the positions $0, 1, \dots, T - 1$ and an even width $d$. Pair $i$, made of dimensions
$2i$ and $2i + 1$ (counting from 0), has the speed $w_i = 1 / 10000^{2i/d}$ and holds
$(\sin(w_i p), \cos(w_i p))$ at position $p$, as in Step 3. Build the table with array
operations, without a Python loop.

In [ ]:
def positions(T, d):
    """T x d sinusoidal position values for positions 0 to T - 1."""
    # TODO
    raise NotImplementedError

In [ ]:
P = positions(3, 4)
print(P.round(3))
assert P.shape == (3, 4), f"expected shape (3, 4), got {P.shape}"
assert np.allclose(P[0], [0, 1, 0, 1]), \
    f"position 0 should be (0, 1, 0, 1), got {P[0]}: the first position is 0, and each pair holds its sine, then its cosine"
assert np.allclose(P[:, 0], np.sin([0, 1, 2])) and np.allclose(P[:, 1], np.cos([0, 1, 2])), \
    "the first pair should hold sin p and cos p, the fast pair of Step 3"
step3 = [[0, 1, 0, 1],
         [0.841471, 0.540302, 0.010000, 0.999950],
         [0.909297, -0.416147, 0.019999, 0.999800]]
assert np.allclose(P, step3, atol=1e-5), "the second pair differs from Step 3's table: with width 4 its speed is 1/100"
P8 = positions(6, 8)
assert P8.shape == (6, 8), f"expected shape (6, 8), got {P8.shape}"
speeds = np.array([1, 0.1, 0.01, 0.001])
assert np.allclose(P8[5, 0::2], np.sin(5 * speeds)) and np.allclose(P8[5, 1::2], np.cos(5 * speeds)), \
    "with width 8 the speeds of the four pairs should be 1, 0.1, 0.01 and 0.001"
print("Part 1 passed")

## Part 2 (coding task): Causal attention with any number of heads

Write `attention(H, heads)`, causal multi-head attention with the setup's tables `W_Q`, `W_K`,
`W_V` and `W_O`, as in Steps 4 to 8. `H` is $T \times d$ with $d = 4$, and `heads` divides
$d$. With $d_k = d / \text{heads}$, head $h$ (counting from 0) uses columns $h d_k$ to
$(h + 1) d_k - 1$ of the queries, keys and values, and divides its scores by $\sqrt{d_k}$.
Return two arrays:

- `M`, the output of the layer, which is the heads' outputs side by side times `W_O`, of
  shape $T \times d$.
- `A`, the weights, of shape `(heads, T, T)`, with `A[h]` the weights of head $h$.

Try to handle all the heads at once, without a Python loop over them.

In [ ]:
def attention(H, heads):
    """Causal multi-head attention. Returns M (T x d) and A (heads x T x T)."""
    T, d = H.shape
    d_k = d // heads
    # TODO
    raise NotImplementedError

In [ ]:
H = np.array([[1.000000, 1.000000, 0.000000, 1.000000],      # "the cat sat" after Step 3
              [0.841471, 1.540302, 1.010000, 0.999950],
              [0.909297, -0.416147, 1.019999, 1.999800]])
M, A = attention(H, 2)
print(A.round(3))
print(M.round(3))
assert A.shape == (2, 3, 3), f"A should have shape (heads, T, T) = (2, 3, 3), got {A.shape}"
assert M.shape == (3, 4), f"M should have shape (T, d) = (3, 4), got {M.shape}"
assert np.allclose(A.sum(axis=-1), 1), "each token's weights should add up to 1: take the softmax along the keys, the last axis"
assert np.allclose(np.triu(A, k=1), 0), "a token must put no weight on the tokens after it"
assert np.allclose(A[0, 2], [0.115, 0.709, 0.175], atol=1e-3), \
    f"head 0 weights of sat are {A[0, 2].round(3)}, Step 6 has (0.115, 0.709, 0.175)"
assert np.allclose(A[1, 2], [0.124, 0.105, 0.771], atol=1e-3), \
    f"head 1 weights of sat are {A[1, 2].round(3)}, Step 6 has (0.124, 0.105, 0.771)"
assert np.allclose(M[2], [0.872, 1.135, 0.893, 1.771], atol=1e-3), \
    f"the output row of sat is {M[2].round(3)}, Step 8 has (0.872, 1.135, 0.893, 1.771)"
H_ran = H.copy()
H_ran[2] = [1.909297, -0.416147, 1.019999, 0.999800]          # "the cat ran"
M_ran, A_ran = attention(H_ran, 2)
assert np.allclose(M_ran[:2], M[:2]) and np.allclose(A_ran[:, :2], A[:, :2]), \
    "changing the third token changed the rows of the first two"
M4, A4 = attention(H, 4)
assert A4.shape == (4, 3, 3), f"with 4 heads A should have shape (4, 3, 3), got {A4.shape}"
assert np.allclose(A4[:, 2], [[0.3366, 0.5840, 0.0794], [0.0616, 0.4645, 0.4739],
                              [0.2777, 0.2218, 0.5006], [0.1066, 0.1065, 0.7869]], atol=1e-3), \
    "with 4 heads of one number each, the weights of sat are wrong: check the divisor and which columns each head gets"
assert np.allclose(M4[2], [0.9002, 0.5799, 0.7346, 1.7867], atol=1e-3), "with 4 heads the output row of sat is wrong"
print("Part 2 passed")

## Part 3 (coding task): From token IDs to next-token probabilities

Write `next_token_probs(ids)`, the whole model of Steps 2 to 12 for a list of token IDs:

1. Look up the rows of `E` and add `positions(T, 4)`.
2. Run `attention` with 2 heads, add its output to the input and apply `layer_norm`.
3. Apply the feed-forward network, `ReLU(Y W_1) W_2`, add its output to its input and apply
   `layer_norm` again.
4. Make the logits with the tied table, $H^{(1)} E^\top$, and return the softmax of each
   row, a $T \times 8$ array whose row $t$ holds the probabilities of the token after
   position $t$.

In [ ]:
def next_token_probs(ids):
    """T x 8 probabilities: row t is the model's prediction for the token after position t."""
    # TODO
    raise NotImplementedError

In [ ]:
p = next_token_probs([1, 2, 3])              # "the cat sat"
print(p.round(3))
assert p.shape == (3, 8), f"expected shape (3, 8), got {p.shape}"
assert np.allclose(p.sum(axis=1), 1), "the 8 probabilities at every position should add up to 1"
assert np.isclose(p[2, 4], 0.741, atol=1e-3), \
    f"after 'the cat sat' the probability of 'on' should be 0.741, as in Step 12, got {p[2, 4]:.3f}"
loss = -np.mean(np.log(p[[0, 1, 2], [2, 3, 4]]))
assert np.isclose(loss, 2.596, atol=1e-3), f"the loss on the targets cat, sat, on is {loss:.3f}, Step 12 has 2.596"
p4 = next_token_probs([1, 2, 3, 4])          # "the cat sat on"
assert np.allclose(p4[:3], p), "adding a fourth token changed the predictions at the first three positions"
assert np.isclose(p4[3, 4], 0.736, atol=1e-3), f"after 'the cat sat on' the probability of 'on' should be 0.736, got {p4[3, 4]:.3f}"
q = next_token_probs([1, 6, 7])              # "the dog ran"
assert np.allclose(q[2], [0.0393, 0.2174, 0.0148, 0.1488, 0.3092, 0.0449, 0.0350, 0.1906], atol=1e-3), \
    "the predictions after 'the dog ran' are wrong"
print("Part 3 passed")

## Part 4 (coding task): Greedy generation

Write `generate(ids, max_new)`. It runs the model on the tokens so far, appends the most
likely token after the last position, and repeats. It stops after `max_new` new tokens, or as
soon as the token it appends is `<end>` (ID 0), which stays in the result. It returns the whole
list of IDs, the prompt included, as a Python list of integers.

In [ ]:
def generate(ids, max_new):
    """Greedy decoding from the prompt ids; returns the prompt plus the new tokens."""
    # TODO
    raise NotImplementedError

In [ ]:
out = generate([1, 2, 3], 4)
assert isinstance(out, list), f"return a Python list, got {type(out).__name__}"
assert all(0 <= i < len(vocab) for i in out), f"every ID should be a token of the vocabulary, 0 to 7, got {out}"
print(" ".join(vocab[i] for i in out))
assert out[:3] == [1, 2, 3], "the result should start with the prompt"
assert len(out) == 7, f"4 new tokens after a 3-token prompt make 7, got {len(out)}"
assert out == [1, 2, 3, 4, 4, 4, 4], f"from 'the cat sat' the model gives {' '.join(vocab[i] for i in out)}"
assert generate([1], 6) == [1, 1, 1, 1, 4, 4, 4], "from 'the' alone the greedy text is wrong: use the prediction after the last position each time"
assert generate([7], 5) == [7, 0], f"from 'ran' the model predicts <end> at once, got {generate([7], 5)}: stop there and keep the <end>"
assert generate([1, 2, 3], 0) == [1, 2, 3], "with max_new = 0 the prompt comes back unchanged"
print("Part 4 passed")

## Solution

### Part 1: Position values for any length and width

In [ ]:
def positions(T, d):
    """T x d sinusoidal position values for positions 0 to T - 1."""
    p = np.arange(T)[:, None]                      # T x 1
    w = 1 / 10000 ** (2 * np.arange(d // 2) / d)   # one speed per pair
    angles = p * w                                 # T x d/2
    P = np.zeros((T, d))
    P[:, 0::2] = np.sin(angles)
    P[:, 1::2] = np.cos(angles)
    return P

**Why it works.** `p * w` multiplies a column of positions by a row of speeds, so broadcasting
gives every angle $w_i p$ at once, one row per position and one column per pair. The sines go
to the even columns and the cosines to the odd ones, so each pair sits side by side as in
Step 3. The exponent is $2i/d$ with $i$ the pair, which runs the speeds from 1 down to
$10000^{-(d-2)/d}$: 1 and 0.01 for width 4, and 1, 0.1, 0.01 and 0.001 for width 8.

### Part 2: Causal attention with any number of heads

In [ ]:
def attention(H, heads):
    """Causal multi-head attention. Returns M (T x d) and A (heads x T x T)."""
    T, d = H.shape
    d_k = d // heads

    def split(X):
        # T x d -> heads x T x d_k: head h gets columns h*d_k to (h+1)*d_k - 1.
        return X.reshape(T, heads, d_k).transpose(1, 0, 2)

    Q, K, V = split(H @ W_Q), split(H @ W_K), split(H @ W_V)
    S = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)          # heads x T x T
    later = np.triu(np.ones((T, T), dtype=bool), k=1)    # True above the diagonal
    A = softmax(np.where(later, -np.inf, S), axis=-1)
    O = A @ V                                            # heads x T x d_k
    M = O.transpose(1, 0, 2).reshape(T, d) @ W_O         # heads side by side
    return M, A

**Why it works.** `reshape(T, heads, d_k)` cuts each row into `heads` consecutive blocks of
`d_k` numbers, which are exactly the columns each head owns, and `transpose(1, 0, 2)` moves the
head to the front, so `Q @ K.transpose(0, 2, 1)` is one $T \times T$ table of scores per head.
Reshaping straight to `(heads, T, d_k)` would also run, but it would fill each head with
numbers from different tokens. The mask sets every score above the diagonal to $-\infty$, so
the softmax gives it a weight of exactly 0. Putting the heads back side by side reverses the
split: `transpose(1, 0, 2)` and then `reshape(T, d)`.

### Part 3: From token IDs to next-token probabilities

In [ ]:
def next_token_probs(ids):
    """T x 8 probabilities: row t is the model's prediction for the token after position t."""
    T = len(ids)
    H = E[ids] + positions(T, 4)              # Steps 2 and 3
    M, _ = attention(H, 2)                    # Steps 4 to 8
    Y = layer_norm(H + M)                     # Step 9
    F = np.maximum(Y @ W_1, 0) @ W_2          # Step 10
    H1 = layer_norm(Y + F)
    return softmax(H1 @ E.T, axis=-1)         # Step 12, with the tied table

**Why it works.** Each line is one step of the chapter, and every step after the lookup keeps
the shape $T \times 4$ until the logits, so the block's pieces fit together in order. The
residual additions use the input of each part, `H + M` and `Y + F`, and LayerNorm works along
the last axis, one token at a time. Because the attention mask hides later tokens and every
other step works on each row alone, row $t$ depends only on the first $t + 1$ tokens, which is
why adding "on" leaves the first three rows exactly as they were.

### Part 4: Greedy generation

In [ ]:
def generate(ids, max_new):
    """Greedy decoding from the prompt ids; returns the prompt plus the new tokens."""
    ids = list(ids)
    for _ in range(max_new):
        p = next_token_probs(ids)
        token = int(np.argmax(p[-1]))         # the prediction after the last position
        ids.append(token)
        if token == 0:                        # <end>
            break
    return ids

**Why it works.** Row $t$ of the probabilities predicts the token after position $t$, so the
next token always comes from the last row, `p[-1]`. The model runs again on the longer list,
with fresh position values for the new length. From "the cat sat" it repeats "on", the loop
Step 13 describes: its weights were chosen for one prediction. From "ran" the most likely token
is `<end>`, so generation stops after one step. A real implementation would keep the keys and
values of earlier tokens in a cache instead of running the model on the whole list each time.